In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

# ============================================================
# CONFIG
# ============================================================

METADATA_DIR = Path("/Users/researcher/Desktop/LSE/AzureData/Accidents/Metadata/Metadata")          # Folder containing metadata .txt files
OUTPUT_CSV = "streetview_metadata.csv"

ANGLES = [0, 90, 180, 270]

# ============================================================
# PROCESS
# ============================================================

rows = []

metadata_files = sorted(METADATA_DIR.glob("*.txt"))

print(f"Found {len(metadata_files):,} metadata files")

for file in metadata_files:

    try:
        with open(file, "r", encoding="utf-8") as f:
            md = json.load(f)

        row_idx = md["row_idx"]

        requested_latitude = md.get("lat")
        requested_longitude = md.get("lng")

        location = md.get("location", {})
        retrieved_latitude = location.get("lat")
        retrieved_longitude = location.get("lng")

        time_utc = md.get("metadata_response", {}).get("date")
        pano_id = md.get("metadata_response", {}).get("pano_id")

        base_name = file.stem  # e.g. "3_90"

        rows.append({
            "image_name": base_name,
            "requested_latitude": requested_latitude,
            "requested_longitude": requested_longitude,
            "retrieved_latitude": retrieved_latitude,
            "retrieved_longitude": retrieved_longitude,
            "time_utc": time_utc,
            "pano_id": pano_id,
        })

    except Exception as e:
        print(f"Skipping {file.name}: {e}")

# ============================================================
# SAVE
# ============================================================

df = pd.DataFrame(rows)

df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved {OUTPUT_CSV}")
print(f"Rows: {len(df):,}")